In [ ]:
#Importing necessary libraries
from sagemaker.huggingface import HuggingFace
import sagemaker
import os
from dotenv import load_dotenv
load_dotenv()

In [ ]:
import os

# Remove proxy vars if not needed
os.environ.pop("HTTP_PROXY", None)
os.environ.pop("HTTPS_PROXY", None)

# Use the certifi-provided CA bundle
os.environ["AWS_CA_BUNDLE"] = "/Volumes/LaCie/Projects_portfolio/NLP/SupportIQ/venv/lib/python3.11/site-packages/certifi/cacert.pem"

# Test AWS call
import boto3
print(boto3.client('sts').get_caller_identity())

In [ ]:
#settinng up AWS profile
os.environ['AWS_PROFILE'] = 'dev-user'

session = sagemaker.Session()

In [ ]:
#Defining hyper parameters
hyper_parameters = {
    'model_id': 'google/flan-t5-base',
    'task_type':'seq2seq',
    'rank': 64,
    'alpha': 128,
    'dropout': 0.05,
    'bias': 'none',
    'lr': 1e-5,
    'epochs': 10,
    'wd': 0.01,
    'logging_steps': 50,
    'batch_size': 32,
    'save_steps': 200,
    'eval_steps': 100,
    'target_module': 'q,k,v',
    'early_stopping': 3,
    'max_length': 64
}

In [ ]:
huggingface_estimator = HuggingFace(
            entry_point='train.py',
            source_dir=os.getenv("SOURCE_DIR"),
            role=os.getenv("ROLE"),
            instance_type='ml.g4dn.xlarge',
            instance_count=1,
            transformers_version='4.49.0',
            py_version='py311',
            pytorch_version='2.5.1',
            hyperparameters = hyper_parameters
)

In [ ]:
huggingface_estimator.fit({'training': 's3://gen-ai-repository/finetuning/flan-t5/data/'}, wait=False)

In [ ]:
job_name = huggingface_estimator.latest_training_job.name

In [ ]:
sm = boto3.client('sagemaker')

In [ ]:
status = sm.describe_training_job(TrainingJobName=job_name)
print("Status:", status['TrainingJobStatus'])

In [ ]:
#After Job complete run this to see logs
# estimator = HuggingFace.attach(training_job_name=job_name)  
# estimator.logs(stream=True, wait=False)

In [ ]:
model_artifact_s3_uri = huggingface_estimator.model_data
print("Model Artifact S3 URI:")
print(model_artifact_s3_uri)